## Consistent Words-based Similarity Matrix

In [1]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
import os
import json
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from gensim.models import FastText

In [2]:
# model_5th = Word2Vec.load("Word2Vec/Exp3/model_5th.model")
# model_12th = Word2Vec.load("Word2Vec/Exp3/model_12th.model")
model_13th = Word2Vec.load("Exp3/model_13th.model")
model_14th = Word2Vec.load("Exp3/model_14th.model")
model_15th = Word2Vec.load("Exp3/model_15th.model")
# model_16th = Word2Vec.load("Word2Vec/Exp3/model_16th.model")
model_18th = Word2Vec.load("Exp3/model_18th.model")
model_19th = Word2Vec.load("Exp3/model_19th.model")
model_20th = Word2Vec.load("Exp3/model_20th.model")

In [3]:
class SimpleCrossLingualComparator:
    def __init__(self, wv_A, wv_B, anchors):
        """
        wv_A: Gensim KeyedVectors object for Space A (model_A.wv)
        wv_B: Gensim KeyedVectors object for Space B (model_B.wv)
        anchors: List of string words to serve as reference points
        """
        self.wv_A = wv_A
        self.wv_B = wv_B
        
        # 1. Filter Anchors
        # We only want anchors that actually exist in BOTH models.
        # We must preserve order so row 0 in Matrix A matches row 0 in Matrix B.
        self.valid_anchors = [w for w in anchors if w in wv_A and w in wv_B]
        
        if not self.valid_anchors:
            raise ValueError("No valid anchors found common to both models!")

        # 2. Build Anchor Matrices directly from the KeyedVectors
        # np.stack turns a list of 1D arrays into a 2D matrix
        self.anchor_matrix_A = np.stack([wv_A[w] for w in self.valid_anchors])
        self.anchor_matrix_B = np.stack([wv_B[w] for w in self.valid_anchors])

        # Pre-calculate norms for the anchors to speed up cosine calculation later
        self.anchor_norms_A = np.linalg.norm(self.anchor_matrix_A, axis=1)
        self.anchor_norms_B = np.linalg.norm(self.anchor_matrix_B, axis=1)

    def _cosine_similarity_vectorized(self, target_vec, anchor_matrix, anchor_norms):
        """
        Computes cosine similarity of target_vec against all anchors.
        """
        # Dot product
        dot_products = np.dot(anchor_matrix, target_vec)
        
        # Norm of target
        norm_target = np.linalg.norm(target_vec)
        
        # Cosine = (A . B) / (|A| * |B|)
        # We add 1e-10 to denominator to avoid division by zero errors
        similarities = dot_products / ((norm_target * anchor_norms) + 1e-10)
        return similarities

    def get_similarity(self, word_A, word_B):
        # Check if words exist in their respective models
        if word_A not in self.wv_A or word_B not in self.wv_B:
            print(f"Warning: '{word_A}' or '{word_B}' not in vocabulary.")
            return 0.0

        # 1. Get vectors directly
        vec_A = self.wv_A[word_A]
        vec_B = self.wv_B[word_B]

        # 2. Create Similarity Profiles
        profile_A = self._cosine_similarity_vectorized(vec_A, self.anchor_matrix_A, self.anchor_norms_A)
        profile_B = self._cosine_similarity_vectorized(vec_B, self.anchor_matrix_B, self.anchor_norms_B)

        # 3. Correlation
        correlation, _ = pearsonr(profile_A, profile_B)
        return correlation

In [4]:
with open("../Data/consistent_keys_2.json", "r", encoding="utf-8") as f:
    consistent_keys = json.load(f)

In [5]:
comparator = SimpleCrossLingualComparator(model_18th.wv, model_20th.wv, consistent_keys)

In [6]:
score_1 = comparator.get_similarity("සතර", "සතර")

In [7]:
score_2 = comparator.get_similarity("මල්", "මල්")

In [8]:
print(f"Self-similarity across spaces: {score_1, score_2}") 

Self-similarity across spaces: (np.float32(0.26802123), np.float32(0.110259466))


In [9]:
with open("../Data/consistent_keys.json", "r", encoding="utf-8") as f:
    consistent_keys = json.load(f)

In [10]:
len(consistent_keys)

80

In [11]:
consistent_keys_years = [13, 14, 15, 18, 19, 20]

In [12]:
results_list = []

In [13]:
# for key in consistent_keys:    
#     n_years = len(consistent_keys_years)
#     similarity_matrix = pd.DataFrame(index=consistent_keys_years, columns=consistent_keys_years, data=np.full((n_years, n_years), np.nan), dtype=float)

#     for i in range(len(consistent_keys_years)):
#         year_a = consistent_keys_years[i]
#         model_a = globals()[f"model_{year_a}th"]
#         similarity_matrix.loc[year_a, year_a] = 1.0

#         for j in range(i+1, len(consistent_keys_years)):
#             year_b = consistent_keys_years[j]
#             model_b = globals()[f"model_{year_b}th"]
#             # print(year_a, year_b)

#             if year_a == year_b:
#                 continue
            
#             comparator = SimpleCrossLingualComparator(model_a.wv, model_b.wv, consistent_keys)
#             score = comparator.get_similarity(key, key)

#             similarity_matrix.loc[year_a, year_b] = score
#             similarity_matrix.loc[year_b, year_a] = score
    
#     similarity_matrix['Key'] = key
#     similarity_matrix = similarity_matrix.reset_index().rename(columns={'index': 'Year_A'})
#     results_list.append(similarity_matrix)
#     # print(similarity_matrix)
#     # break
# final_similarity_df = None
# final_similarity_df = pd.concat(results_list, ignore_index=True)

#             # print(f"Key: {key} | {year_a}th to {year_b}th similarity: {score}")
    

In [14]:
# ---------------------------------------------------------
# UPDATE: Lemma-Aggregated Similarity Logic
# ---------------------------------------------------------
from collections import defaultdict

# 1. Load Data for Lemma Aggregation
print("Loading lemma and token data...")
with open('../data/century_lemmas.json', 'r', encoding='utf-8') as f:
    century_lemmas = json.load(f)

with open('../data/century_tokens.json', 'r', encoding='utf-8') as f:
    century_tokens = json.load(f)

# 2. Build Lemma -> Words Map
century_lemma_words = defaultdict(lambda: defaultdict(set))
valid_centuries = sorted(century_lemmas.keys(), key=lambda x: int(x))
valid_centuries.remove('5')
valid_centuries.remove('12')
valid_centuries.remove('16')
print(valid_centuries) #= [13, 14, 15, 16, 18, 19, 20]

for century in valid_centuries:
    l_sentences = century_lemmas[century]
    t_sentences = century_tokens[century]
    for l_sent, t_sent in zip(l_sentences, t_sentences):
        for l_word, t_word in zip(l_sent, t_sent):
            if l_word and t_word:
                 century_lemma_words[century][l_word].add(t_word)

# 3. Find Common Lemmas (Intersection serves as NEW consistent_keys)
common_lemmas = None
for century in valid_centuries:
    keys = set(century_lemma_words[century].keys())
    if common_lemmas is None:
        common_lemmas = keys
    else:
        common_lemmas = common_lemmas.intersection(keys)

consistent_keys = sorted(list(common_lemmas))
print(f"Algorithm updated to use {len(consistent_keys)} common lemmas as keys.")

# 4. Define Proxy Class for Aggregation
class LemmaVectorProxy:
    def __init__(self, original_model, lemma_map):
        self.wv = original_model.wv # Access .wv from the model
        self.lemma_map = lemma_map
        
    def __contains__(self, key):
        if key not in self.lemma_map: return False
        words = self.lemma_map[key]
        return any(w in self.wv for w in words)
        
    def __getitem__(self, key):
        words = self.lemma_map.get(key, [])
        valid_vectors = [self.wv[w] for w in words if w in self.wv]
        if not valid_vectors:
            raise KeyError(f"Lemma '{key}' has no valid word forms in model.")
        return np.mean(valid_vectors, axis=0)

# 5. Build Proxies for all Loaded Models
model_map = {}
# Helper to grab globals safely
current_globals = globals()
for year in consistent_keys_years:
    model_var_name = f"model_{year}th"
    if model_var_name in current_globals:
        model_map[str(year)] = current_globals[model_var_name]

proxies = {}
for year_str, model in model_map.items():
    if year_str in century_lemma_words:
        proxies[year_str] = LemmaVectorProxy(model, century_lemma_words[year_str])

print(f"Proxies built for centuries: {list(proxies.keys())}")

# 6. Execute Alignment Loop
results_list = []

for key in consistent_keys:    
    n_years = len(consistent_keys_years)
    similarity_matrix = pd.DataFrame(index=consistent_keys_years, columns=consistent_keys_years, data=np.full((n_years, n_years), np.nan), dtype=float)

    for i in range(len(consistent_keys_years)):
        year_a = consistent_keys_years[i]
        str_year_a = str(year_a)
        
        # Use Proxy if available
        if str_year_a not in proxies: continue
        # NOTE: Comparator expects .wv interface. Our Proxy mimics it enough for __getitem__ and __contains__
        # But Comparator constructor does 'if w in wv'. Proxy supports this via __contains__
        # Comparator does 'wv[w]'. Proxy supports this via __getitem__
        
        proxy_a = proxies[str_year_a]
        similarity_matrix.loc[year_a, year_a] = 1.0

        for j in range(i+1, len(consistent_keys_years)):
            year_b = consistent_keys_years[j]
            str_year_b = str(year_b)
            
            if str_year_b not in proxies: continue
            proxy_b = proxies[str_year_b]

            if year_a == year_b:
                continue
            
            # Create Comparator with Proxies
            # 'consistent_keys' is now the list of ALL common lemmas.
            # The comparator will use these lemmas to build the anchor matrix.
            # It will ask proxy['lemma_X'], proxy returns mean vector.
            comparator = SimpleCrossLingualComparator(proxy_a, proxy_b, consistent_keys)
            
            # Get similarity for the CURRENT target lemma 'key'
            if key in proxy_a and key in proxy_b:
                score = comparator.get_similarity(key, key)
                similarity_matrix.loc[year_a, year_b] = score
                similarity_matrix.loc[year_b, year_a] = score
    
    similarity_matrix['Key'] = key
    similarity_matrix = similarity_matrix.reset_index().rename(columns={'index': 'Year_A'})
    results_list.append(similarity_matrix)

final_similarity_df = None
if results_list:
    final_similarity_df = pd.concat(results_list, ignore_index=True)
    print(f"Done. Computed {len(final_similarity_df)} rows.")
else:
    print("No results.")


Loading lemma and token data...
['13', '14', '15', '18', '19', '20']
Algorithm updated to use 261 common lemmas as keys.
Proxies built for centuries: ['13', '14', '15', '18', '19', '20']
Done. Computed 1566 rows.


In [15]:
len(consistent_keys)

261

In [16]:
century_lemma_words

defaultdict(<function __main__.<lambda>()>,
            {'13': defaultdict(set,
                         {'අප': {'අප', 'අපගේ', 'අපට', 'අපි', 'අපේ'},
                          'බුදුන්': {'බුදුන්', 'බුදුන්ගේ', 'බුදුන්ට'},
                          'සාරාස': {'සාරාසකි'},
                          'කප්සුවහස': {'කප්සුවහස්'},
                          'මතු': {'මතු', 'මතුයෙහි'},
                          'කුලුණුනු': {'කුලුණුනු'},
                          'වණ': {'වණින්'},
                          'යුත්': {'යුත්'},
                          'මහත': {'මහත', 'මහතෙ'},
                          'ව': {'නොව',
                           'නොවන',
                           'නොවන්නේය',
                           'නොවූ',
                           'නොවෙයි',
                           'නොවේ',
                           'ව',
                           'වන්නට',
                           'වන්නීය',
                           'වන්නේය',
                           'වයො',
                           'විනි',
      

In [17]:
final_similarity_df.head(100)

,Year_A,13,14,15,18,19,20,Key
0,13,1.000000,0.205615,0.358888,0.189544,0.387248,0.350963,අක්ෂර
1,14,0.205615,1.000000,0.298748,0.181362,0.234598,0.263758,අක්ෂර
2,15,0.358888,0.298748,1.000000,0.218153,0.440864,0.353213,අක්ෂර
3,18,0.189544,0.181362,0.218153,1.000000,0.064707,0.092998,අක්ෂර
4,19,0.387248,0.234598,0.440864,0.064707,1.000000,0.403157,අක්ෂර
...,...,...,...,...,...,...,...,...
95,20,0.596439,0.241978,0.418304,0.063200,0.573098,1.000000,ඇති
96,13,1.000000,0.286716,0.334922,0.202651,0.485583,0.471037,ඇතිව
97,14,0.286716,1.000000,0.328471,0.171412,0.193199,0.290263,ඇතිව
98,15,0.334922,0.328471,1.000000,0.194714,0.310949,0.337484,ඇතිව


In [18]:
final_similarity_df.to_csv("../Data/Word2Vec_Cross_Lingual_Similarities_v1.csv", index=False)

In [19]:
unique_keys = final_similarity_df['Key'].unique()

In [20]:
len(unique_keys)

261

In [21]:
font_path = '../Fonts/Noto_Sans_Sinhala/NotoSansSinhala-VariableFont_wdth,wght.ttf'
fm.fontManager.addfont(font_path)
sinhala_font_name = fm.FontProperties(fname=font_path).get_name()
fm._load_fontmanager(try_read_cache=False)

Matplotlib is building the font cache; this may take a moment.


In [22]:
output_dir = '../Data/Word_Similarity_Heatmaps_Method_1_WV'

In [23]:
for key in unique_keys:
    key_df = final_similarity_df[final_similarity_df['Key'] == key].copy()
    year_cols = [col for col in key_df.columns if col not in ['Year_A', 'Key']]
    heatmap_data = key_df.set_index('Year_A')[year_cols].apply(pd.to_numeric, errors='coerce')
    plt.figure(figsize=(9, 7))

    sns.heatmap(
        heatmap_data, 
        annot=True,              # Show the similarity value on the cell
        fmt=".3f",               # Format the annotation to 3 decimal places
        cmap="YlGnBu",           # A nice sequential color map
        linewidths=0.5,          # Line width between cells
        linecolor='white',       # Color of the lines
        cbar_kws={'label': 'Cross-Year Word Similarity Score'} # Label for the color bar
    )

    plt.title(f'Cross-Year Cosine Similarity for Word: "{key}"', fontsize=14, fontproperties=sinhala_font_name)
    plt.xlabel('Year B (Model Version)', fontsize=12, fontproperties=sinhala_font_name)
    plt.ylabel('Year A (Model Version)', fontsize=12, fontproperties=sinhala_font_name)

    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout() # Adjust plot to fit all elements
    # plt.show()
    safe_key = str(key).replace(' ', '_')
    filename = os.path.join(output_dir, f"{safe_key}.png")

    plt.savefig(
        filename, 
        dpi=300, # High resolution for better quality
        bbox_inches='tight' # Trims unnecessary white space
    )

    plt.close()
    print(f"Saved: {filename}")



Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අක්ෂර.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අග්ර.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අත.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අතුර.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අන්ධකාර.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අන්ය.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අනෙක.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අප.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අර්ථ.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අව.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/අස.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/ආ.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/ආදි.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/ආදී.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/ඇත්තේ.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_1_WV/ඇති.png
Saved: ../Data/Word_Similarity_He

In [24]:
df_long = final_similarity_df.melt(id_vars=['Key', 'Year_A'], var_name='Year_B', value_name='Value')
df_long['Year_B'] = pd.to_numeric(df_long['Year_B'])
df_long['feature_pair'] = df_long['Year_A'].astype(str) + '_' + df_long['Year_B'].astype(str)
df_lags_only = df_long[df_long['Year_A'] < df_long['Year_B']]
piv_similarity_df = df_lags_only.pivot(index='Key', columns='feature_pair', values='Value')
piv_similarity_df['Mean_Similarity'] = piv_similarity_df.mean(axis=1, numeric_only=True)
piv_similarity_df = piv_similarity_df.reset_index()

In [25]:
piv_similarity_df.head()

feature_pair,Key,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20,Mean_Similarity
0,අක්ෂර,0.205615,0.358888,0.189544,0.387248,0.350963,0.298748,0.181362,0.234598,0.263758,0.218153,0.440864,0.353213,0.064707,0.092998,0.403157,0.269588
1,අග්ර,0.156931,0.351365,0.106995,0.415872,0.431296,0.302190,0.222418,0.173024,0.129298,0.258664,0.318343,0.405192,0.193872,0.215517,0.402333,0.272221
2,අත,0.355007,0.491921,0.251210,0.534838,0.561143,0.358868,0.062001,0.335225,0.299432,0.182008,0.485910,0.475464,0.164512,0.172337,0.599956,0.355322
3,අතුර,0.404096,0.472359,0.189186,0.462015,0.494712,0.313054,0.113609,0.260260,0.313929,0.162410,0.484190,0.390064,0.137248,0.122454,0.512063,0.322110
4,අන්ධකාර,0.242446,0.300084,0.153567,0.439383,0.466479,0.334248,0.157511,0.257262,0.260363,0.133613,0.425323,0.384446,0.159965,0.177402,0.457100,0.289946


In [26]:
piv_similarity_df['Mean_Similarity'].describe()

count    261.000000
mean       0.302341
std        0.051061
min        0.181981
25%        0.264332
50%        0.302401
75%        0.341549
max        0.445767
Name: Mean_Similarity, dtype: float64

In [27]:
# --- The Solution Code ---
# Melt to long format
df_long = piv_similarity_df.melt(id_vars='Key', var_name='feature', value_name='Value')

# Drop NaNs
df_long = df_long.dropna()

# Get top 10 smallest per Key
top_10_per_key = df_long.groupby('feature', group_keys=False).apply(lambda x: x.nsmallest(10, 'Value'))

/var/folders/p0/y_n4880x7cjbhx11tzvvqpzh0000gn/T/ipykernel_14828/1399102897.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_10_per_key = df_long.groupby('feature', group_keys=False).apply(lambda x: x.nsmallest(10, 'Value'))


In [28]:
print(top_10_per_key[top_10_per_key['feature'] == '18_20'])

            Key feature     Value
3599      වනාහි   18_20 -0.042325
3529        පිට   18_20  0.000228
3507     නොයෙක්   18_20  0.004275
3629   සම්පුර්ණ   18_20  0.011376
3515  පරිද්දෙන්   18_20  0.017893
3535      පුරුෂ   18_20  0.027442
3595         වට   18_20  0.035118
3648         හා   18_20  0.037533
3579         රට   18_20  0.040281
3443         කා   18_20  0.045052


In [29]:
piv_similarity_df[piv_similarity_df['Key'] == 'සතර']

feature_pair,Key,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20,Mean_Similarity
229,සතර,0.199738,0.413153,0.120047,0.334899,0.434497,0.212772,0.146506,0.165284,0.226747,0.104583,0.425936,0.396432,0.179999,0.220355,0.495369,0.271754


In [30]:
piv_similarity_df.describe()

feature_pair,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20,Mean_Similarity
count,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000,261.000000
mean,0.279997,0.405479,0.196060,0.453468,0.465706,0.283341,0.132732,0.251534,0.234435,0.182133,0.424443,0.390881,0.174297,0.149099,0.511517,0.302341
std,0.072896,0.095081,0.063058,0.101437,0.100035,0.069426,0.039602,0.068531,0.063089,0.055465,0.092496,0.088973,0.055668,0.051698,0.107429,0.051061
min,0.037792,0.141147,-0.008482,0.169569,0.198986,0.022461,0.002400,0.036702,0.084413,0.004858,0.180623,0.160319,0.047645,-0.042325,0.246011,0.181981
25%,0.228989,0.331681,0.159262,0.376180,0.396365,0.236678,0.108360,0.207118,0.189517,0.149482,0.350363,0.319621,0.137825,0.119481,0.429966,0.264332
50%,0.287133,0.408856,0.202651,0.450275,0.466921,0.287864,0.134926,0.258703,0.235421,0.192228,0.419911,0.390694,0.174417,0.153956,0.509103,0.302401
75%,0.330646,0.484456,0.241963,0.535545,0.550021,0.334997,0.162021,0.297734,0.284662,0.220850,0.497288,0.451089,0.209176,0.184005,0.599444,0.341549
max,0.459490,0.646229,0.340879,0.684455,0.668533,0.446190,0.222418,0.389476,0.374006,0.286903,0.652693,0.585770,0.331577,0.252904,0.739777,0.445767


In [31]:
#Only the one's considered for analysis
century_models = {
    13: model_13th, 
    14: model_14th, 
    15: model_15th, 
    # 16: model_16th, 
    18: model_18th, 
    19: model_19th, 
    20: model_20th
}

In [32]:
# def get_neighbors_from_aligned_models(df, models_dict, threshold_std=0.5, top_n=10):
#     results = []
#     # Use 'Key' as index per your piv_df_org structure
#     work_df = df.set_index('Key') if 'Key' in df.columns else df.copy()
    
#     # Identify comparison columns (e.g., '15_18', '18_19')
#     comparison_cols = [c for c in work_df.columns if isinstance(c, str) and '_' in c]
    
#     for col in comparison_cols:
#         parts = col.split('_')
#         try:
#             y1, y2 = int(parts[0]), int(parts[1])
#         except: continue
        
#         if y1 not in models_dict or y2 not in models_dict: 
#             continue

#         m1 = models_dict[y1]
#         m2 = models_dict[y2]

#         # Calculate Dip Threshold
#         valid_vals = work_df[col][work_df[col] > 0]
#         if valid_vals.empty: continue
#         cutoff = valid_vals.mean() - (threshold_std * valid_vals.std())
        
#         # Identify words that dipped
#         dipped_words = valid_vals[valid_vals < cutoff]
        
#         for word, score in dipped_words.items():
#             try:
#                 # 1. Safety Check: Ensure word exists in both vocabularies
#                 if word not in m1.wv or word not in m2.wv:
#                     continue
                
#                 # 2. Safety Check: Ensure top_n doesn't exceed vocabulary size
#                 # This is the most common cause of the IndexError you saw
#                 v1_limit = min(top_n, len(m1.wv.index_to_key) - 1)
#                 v2_limit = min(top_n, len(m2.wv.index_to_key) - 1)
                
#                 # 3. Retrieve neighbors
#                 # We use the Word index explicitly to avoid index range issues
#                 n1 = [n[0] for n in m1.wv.most_similar(word, topn=v1_limit)]
#                 n2 = [n[0] for n in m2.wv.most_similar(word, topn=v2_limit)]
                
#                 results.append({
#                     "Word": word,
#                     "Period": col,
#                     "Similarity": round(score, 4),
#                     f"Neighbors_{y1}": ", ".join(n1),
#                     f"Neighbors_{y2}": ", ".join(n2)
#                 })
#             except (IndexError, KeyError):
#                 # If a specific word still causes an index issue, skip it
#                 continue
                
#     return pd.DataFrame(results)

In [33]:
# dip_neighbors_df = get_neighbors_from_aligned_models(piv_similarity_df, century_models)

In [34]:
# len(dip_neighbors_df)

In [35]:
# dip_neighbors_df.head(20)

In [36]:
# dip_neighbors_df['Word'].unique()

In [37]:
# dip_neighbors_df[dip_neighbors_df['Word'] == 'මංගල']

In [38]:
# piv_similarity_df[piv_similarity_df['Key'] == 'මංගල']

In [39]:
# dip_neighbors_df[dip_neighbors_df['Word'] == 'කිරි']

In [40]:
# dip_neighbors_df['Similarity'].describe()

In [41]:
# dip_neighbors_df.to_csv('../Data/fasttext_dip_neighbors_similarity_matrix.csv', index=False, encoding='utf-8-sig')

In [42]:
def get_semantic_drift_report(df, models_dict, threshold_std=0.8, top_n=10):
    results = []
    # Ensure 'Key' is the index
    work_df = df.set_index('Key') if 'Key' in df.columns else df.copy()
    
    # Identify all comparison columns (13_14, 13_15, 14_15, etc.)
    cols = [c for c in work_df.columns if isinstance(c, str) and '_' in c]
    
    for col in cols:
        parts = col.split('_')
        try:
            y_start, y_end = int(parts[0]), int(parts[1])
        except: continue
        
        if y_start not in models_dict or y_end not in models_dict:
            continue

        m_start = models_dict[y_start]
        m_end = models_dict[y_end]

        # Calculate a dynamic cutoff for this specific period
        series = work_df[col][work_df[col] > 0]
        if series.empty: continue
        
        cutoff = series.mean() - (threshold_std * series.std())
        dips = series[series < cutoff]
        
        for word, score in dips.items():
            try:
                # 1. Check vocab existence
                if word not in m_start.wv or word not in m_end.wv:
                    continue
                
                # 2. Dynamic top_n check to prevent IndexError
                # If vocab is smaller than requested top_n, adjust it
                v_start_size = len(m_start.wv.index_to_key)
                v_end_size = len(m_end.wv.index_to_key)
                safe_n = min(top_n, v_start_size - 1, v_end_size - 1)
                
                if safe_n < 1: continue

                # 3. Retrieve neighbors with explicit error handling for the Gensim bug
                try:
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                except IndexError:
                    # If Gensim's index is still broken, force a refresh and try one last time
                    m_start.wv.fill_norms(force=True)
                    m_end.wv.fill_norms(force=True)
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                
                n_start = [n[0] for n in n_start_raw]
                n_end = [n[0] for n in n_end_raw]
                
                results.append({
                    "Word": word,
                    "Period": col,
                    "Similarity": round(score, 4),
                    f"Neighbors_{y_start}": ", ".join(n_start),
                    f"Neighbors_{y_end}": ", ".join(n_end)
                })
            except Exception:
                # Catch-all for any remaining weirdness with specific words
                continue

    return pd.DataFrame(results)

In [43]:
# Execute
semantic_report = get_semantic_drift_report(piv_similarity_df, century_models, threshold_std=1.5)
semantic_report = semantic_report.sort_values(by=['Period', 'Similarity'])
display(semantic_report.head(30))

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
8,යයි,13_14,0.0378,"අභිප්රෙතයි, විදර්ශනාවට, කාරණයෙන්, විකුර්පණය, ග...","යන්, සේ, නුවරෙක්, එක්තරා, වූයේය, කීර්ණ, ගිරිතට...",NaN,NaN,NaN,NaN
1,ආදී,13_14,0.0711,"ප්රතොලි, ශර්කරා, නගර, ව්යාහත, ප්රාකාර, උපදි, අ...","ඔහු, දත්, පූර්වයෙහි, නන්දෙඩවීම්, පැමිණියා, උපද...",NaN,NaN,NaN,NaN
9,සකල,13_14,0.0815,"චිත්තය, වාසීන්, මුළුලෝ, සම්භාවනීය, මෞදගල්යායන,...","කුලාම්බර, නෙත්රොත්සව, ගණාභරණ, සත්ත්ව, දෙහ, රස්...",NaN,NaN,NaN,NaN
2,කර්ම,13_14,0.0945,"කෙශයෙහි, කාය, ස්ථාන, ථාවරය, ගන්ධය, චර්මානන්තරග...","සක්දෙව්, විශ්ව, අඳුන්, දිවිසම්, නියොග, ලත්, ශා...",NaN,NaN,NaN,NaN
10,සහිත,13_14,0.1191,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ, මේලෙසින්, ...","වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්...",NaN,NaN,NaN,NaN
4,ගෙණ,13_14,0.1200,"දැනැඳින, බොරු, නොගන්මි, යනා, පියවී, දීන්, යන්න...","පරසතු, මදාරා, ඉවූ, පුබුදුවා, තැනීමට, පිටට, කල්...",NaN,NaN,NaN,NaN
5,නමැති,13_14,0.1296,"අවුල, අෂ්ටශත, ජාල, ජාලිනී, විචරිත, නමෙකියි, තෘ...","සන්තු, සෙක්හි, දව, ධාරා, ඊටව, ආකාශචාරීව, වෙසය,...",NaN,NaN,NaN,NaN
6,පරිද්දෙන්,13_14,0.1328,"භාණ්ඩ, සංකීර්ණයද, ගරු, අධිගතානිසංස, අවෙහංගි, ප...","පිළිගෙණ, රජගහ, සතියක්, එලවන, පර්වතච්ඡාවෙහි, රජ...",NaN,NaN,NaN,NaN
12,හා,13_14,0.1518,"ශස්, රූපශ්රී, සඳගණ, නිති, දසමසක්, වාසීන්, මුළු...","ගෙණු, දෙශනා, පෙලෙහි, පැමිණියා, වෙශයක්, සෝවාන්,...",NaN,NaN,NaN,NaN
0,අග්ර,13_14,0.1569,"පාර්ෂණිය, තලය, ඔබන, කලැ, පාර්ශවය, උපස්ථායක, ජි...","බොධිවජ්රාශන, මව්පිය, සාගරයෙන්, දායක, උපදනා, ගො...",NaN,NaN,NaN,NaN


In [44]:
semantic_report.describe()

,Similarity
count,229.000000
mean,0.151175
std,0.094333
min,0.000200
25%,0.071200
50%,0.129200
75%,0.243100
max,0.346100


In [45]:
semantic_report.to_csv('../Data/Word2Vec_dip_neighbors_similarity_matrix.csv', index=False, encoding='utf-8-sig')

In [1]:
import pandas as pd
import numpy as np

In [2]:
semantic_report = pd.read_csv('../Data/Word2Vec_dip_neighbors_similarity_matrix.csv', encoding='utf-8-sig')

In [3]:
semantic_report.head()

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,යයි,13_14,0.0378,"අභිප්රෙතයි, විදර්ශනාවට, කාරණයෙන්, විකුර්පණය, ග...","යන්, සේ, නුවරෙක්, එක්තරා, වූයේය, කීර්ණ, ගිරිතට...",NaN,NaN,NaN,NaN
1,ආදී,13_14,0.0711,"ප්රතොලි, ශර්කරා, නගර, ව්යාහත, ප්රාකාර, උපදි, අ...","ඔහු, දත්, පූර්වයෙහි, නන්දෙඩවීම්, පැමිණියා, උපද...",NaN,NaN,NaN,NaN
2,සකල,13_14,0.0815,"චිත්තය, වාසීන්, මුළුලෝ, සම්භාවනීය, මෞදගල්යායන,...","කුලාම්බර, නෙත්රොත්සව, ගණාභරණ, සත්ත්ව, දෙහ, රස්...",NaN,NaN,NaN,NaN
3,කර්ම,13_14,0.0945,"කෙශයෙහි, කාය, ස්ථාන, ථාවරය, ගන්ධය, චර්මානන්තරග...","සක්දෙව්, විශ්ව, අඳුන්, දිවිසම්, නියොග, ලත්, ශා...",NaN,NaN,NaN,NaN
4,සහිත,13_14,0.1191,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ, මේලෙසින්, ...","වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්...",NaN,NaN,NaN,NaN


In [4]:
sorted_report = semantic_report.sort_values(by='Similarity', ascending=True)

In [5]:
sorted_report.head(6)

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
196,පිට,18_20,0.0002,NaN,NaN,NaN,"දෙමැද්ද, පිරී, පෝරුවක්, වෙනසක්, පිටි, පටලයක්, ...",NaN,"ඇයා, ගගා, කැමැත්ත, මගින්, සභාවක්, හාරා, ප්රධාන..."
25,මෙසේ,13_18,0.0021,"අප්රතිහත, තාය, සරෙහි, පළමුවෙනි, බන්ධනය, ධුතාංග...",NaN,NaN,"දික්ව, ලුහුඬුව, උවමනාතන්හී, සිහින්, තිබෙයි, දළ...",NaN,NaN
88,නොයෙක්,14_18,0.0024,NaN,"ආරාධනා, කළ, නැගී, සහම්පතී, බ්රහ්මයා, සති, ඡන්න...",NaN,"සාදන, පණ්ඩිතයන්, වරම්භීරාදී, නිරාකූලව, රක්ෂිත,...",NaN,NaN
197,නොයෙක්,18_20,0.0043,NaN,NaN,NaN,"සාදන, පණ්ඩිතයන්, වරම්භීරාදී, නිරාකූලව, රක්ෂිත,...",NaN,"සමාජයෙහි, දරණ, අන්දමේ, රත්නජාති, තැනැත්තිය, ගම..."
134,එම,15_18,0.0049,NaN,NaN,"තුඩ, දනන්, සැදූ, මුඛයක්, පුරුෂ, පැවතම, දෙතිස්,...","අන්ය, නැත්තේ, යයිද, අසාස්වත, මරණින්ම, පුද්ගලයන...",NaN,NaN
198,සම්පුර්ණ,18_20,0.0114,NaN,NaN,NaN,"ස්ඛන්ධයයන, දෙපාපිටය, ස්ථානයෙහි, උත්සන්නව, කූටය...",NaN,"දක්වන්නේ, වාක්යයෙකි, අදහසක්, ගැබ්ව, පලල්, අර්ථ..."


In [6]:
semantic_report['Similarity'] = semantic_report['Similarity'].round(4)
split_periods = semantic_report['Period'].str.split('_', expand=True)
ref_num = split_periods[0]
target_num = split_periods[1]

semantic_report['Reference Century'] = ref_num + 'th'
semantic_report['Target Century'] = target_num + 'th'

In [7]:
def get_top_5_from_col(row, century_num):
    col_name = f'Neighbors_{century_num}'
    
    # Check if the column exists and has data for this row
    if col_name in row.index and pd.notna(row[col_name]) and row[col_name] != '':
        # Split by comma, clean up spaces, and grab the first 5 words
        words = [w.strip() for w in str(row[col_name]).split(',') if w.strip()]
        return ', '.join(words[:5])
    
    return '' # Return empty if no data found

# 4. Extract neighbors dynamically for both Reference and Target
semantic_report['Ref Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[0]), axis=1
)

semantic_report['Target Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[1]), axis=1
)


In [8]:
final_df = semantic_report[[
    'Word', 
    'Similarity',
    'Reference Century', 
    'Ref Century Neighbors', 
    'Target Century', 
    'Target Century Neighbors'
]].copy()

In [9]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
0,යයි,0.0378,13th,"අභිප්රෙතයි, විදර්ශනාවට, කාරණයෙන්, විකුර්පණය, ග...",14th,"යන්, සේ, නුවරෙක්, එක්තරා, වූයේය"
1,ආදී,0.0711,13th,"ප්රතොලි, ශර්කරා, නගර, ව්යාහත, ප්රාකාර",14th,"ඔහු, දත්, පූර්වයෙහි, නන්දෙඩවීම්, පැමිණියා"
2,සකල,0.0815,13th,"චිත්තය, වාසීන්, මුළුලෝ, සම්භාවනීය, මෞදගල්යායන",14th,"කුලාම්බර, නෙත්රොත්සව, ගණාභරණ, සත්ත්ව, දෙහ"
3,කර්ම,0.0945,13th,"කෙශයෙහි, කාය, ස්ථාන, ථාවරය, ගන්ධය",14th,"සක්දෙව්, විශ්ව, අඳුන්, දිවිසම්, නියොග"
4,සහිත,0.1191,13th,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ",14th,"වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්තු"


In [10]:
final_df = final_df.sort_values(by='Similarity', ascending=True)

In [11]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
196,පිට,0.0002,18th,"දෙමැද්ද, පිරී, පෝරුවක්, වෙනසක්, පිටි",20th,"ඇයා, ගගා, කැමැත්ත, මගින්, සභාවක්"
25,මෙසේ,0.0021,13th,"අප්රතිහත, තාය, සරෙහි, පළමුවෙනි, බන්ධනය",18th,"දික්ව, ලුහුඬුව, උවමනාතන්හී, සිහින්, තිබෙයි"
88,නොයෙක්,0.0024,14th,"ආරාධනා, කළ, නැගී, සහම්පතී, බ්රහ්මයා",18th,"සාදන, පණ්ඩිතයන්, වරම්භීරාදී, නිරාකූලව, රක්ෂිත"
197,නොයෙක්,0.0043,18th,"සාදන, පණ්ඩිතයන්, වරම්භීරාදී, නිරාකූලව, රක්ෂිත",20th,"සමාජයෙහි, දරණ, අන්දමේ, රත්නජාති, තැනැත්තිය"
134,එම,0.0049,15th,"තුඩ, දනන්, සැදූ, මුඛයක්, පුරුෂ",18th,"අන්ය, නැත්තේ, යයිද, අසාස්වත, මරණින්ම"


In [12]:
final_df['Ref Century Neighbors'] = '\\iffalse ' + final_df['Ref Century Neighbors'].astype(str) + ' \\fi'
final_df['Target Century Neighbors'] = '\\iffalse ' + final_df['Target Century Neighbors'].astype(str) + ' \\fi'

In [15]:
float_formatter = lambda x: f"{x:.4f}".rstrip('0').rstrip('.') if isinstance(x, float) else str(x)

In [17]:
final_df.to_csv('../Data/Word2Vec_dip_neighbors_similarity_matrix_SMA_final.csv', index=False, encoding='utf-8-sig')    

In [16]:
final_df.head(6).to_latex('../Data/Word2Vec_Semantic_Drift_Report_SMA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})

In [22]:
selected_words = ['පුරුෂ', 'සැප', 'චන්ද්රයා']
filtered_df = final_df[final_df["Word"].isin(selected_words)]

In [23]:
cond_others = final_df["Word"].isin(["සැප", "චන්ද්රයා"])

# 2. Condition specifically for 'පුරුෂ' in the 18th to 20th century
cond_purusha_18_20 = (
    (final_df["Word"] == "පුරුෂ") 
    & (final_df["Reference Century"] == "18th") 
    & (final_df["Target Century"] == "20th")
)

# 3. Combine both conditions using the OR (|) operator
filtered_df = final_df[cond_others | cond_purusha_18_20]

In [24]:
filtered_df = filtered_df.sort_values(by='Similarity', ascending=True)

In [25]:
filtered_df

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
200,පුරුෂ,0.0274,18th,"\iffalse හාවිතානු, අසූ, කේතුමාලා, ප්රභානු, පුර...",20th,"\iffalse ජරා, මරණ, මිදුනු, දැන්මද, එකක්හු \fi"
79,චන්ද්රයා,0.1379,14th,"\iffalse උදාකරවා, යන්හි, ස්ථාන, වෛනෙජන, කෞම \fi",15th,"\iffalse ධීතෘවූ, පුණ්ණ, උලකුඩය, දූ, දේවී \fi"
85,සැප,0.1755,14th,"\iffalse දුක්, වටනේය, සසරෙහි, අමෘත, ව්යාධී \fi",15th,"\iffalse සෞඛ්ය, ප්රවෘත්ත, මිත්රොන්තමයෝ, දෙකෙහි..."


In [26]:
filtered_df.to_latex('../Data/Word2Vec_Semantic_Drift_Report_SMA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})